### Collab Code at:

<a href="https://colab.research.google.com/github/tikadatta2005/CSC60904-Deep-Learning-T3/blob/main/experiments/DeepLearning_GroupAssignment_Yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

### Basic Environmental Config

In [ ]:
IMAGE_SIZE = 112
BATCH_SIZE = 32
EPOCHS = 30
SEED = 41

### Model Training
- Load pretrained YOLOv8 Nano classification model (transfer learning)
- Apply augmentation suited to hand-sign images (no left-right flip — it would change which character a hand shape represents)
- Train and save

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n-cls.pt")
print("Pretrained YOLOv8 Nano classification model loaded.")

In [ ]:
# path of models and data
project_path = Path(
    "../documentations/experiments"
)

In [ ]:
import pandas as pd

In [ ]:
results = model.train(
    data="../datasets/final_dataset",
    imgsz=112,
    batch=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,

    # Geometric augmentation
    degrees=10,          # Random rotation: approximately ±10°
    translate=0.05,      # Random translation: ±5%
    scale=0.1,           # Random scale: approximately 0.9× to 1.1×

    # Color augmentation
    hsv_h=0.0,           # No hue change
    hsv_s=0.0,           # No saturation change
    hsv_v=0.2,           # Brightness-like variation

    # No flipping
    fliplr=0.0,
    flipud=0.0,

    # Disable other augmentation methods
    shear=0.0,
    perspective=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    # Save every epoch
    save=True,
    save_period=1,

    project=project_path,
    name="anuj-yolo"
)

print("Training completed.")

Model trained for 30 epochs with augmentation tuned for hand-gesture images; results saved to `/content/NSL_YOLOv8_Results/YOLOv8_NSLv2`.


### Training Performance Analysis
- the model creates metrics but doesnot provide all 4 metrics every epoch
- so each epoch will be loaded
- for each epoch it will be tested on training set and val set to observe model performance

In [ ]:
torch.manual_seed(SEED)
# create train and val set for inference
train_transforms = transforms.Compose([
     transforms.Resize((112, 112)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
])

transform = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor()
])

train_data_set = datasets.ImageFolder("../datasets/final_dataset/train", transform=train_transforms)
val_data_set = datasets.ImageFolder("../datasets/final_dataset/val", transform=transform)

train = DataLoader(
    train_data_set, 
    batch_size=32, 
    num_workers=4, 
    persistent_workers=True, 
    pin_memory=True, 
    shuffle=True
    )

val = DataLoader(
    val_data_set,
    batch_size=32, 
    num_workers=4, 
    persistent_workers=True, 
    pin_memory=True, 
)

In [ ]:
weights_dir = project_path / "anuj-yolo" / "weights"

In [ ]:
# load all checkpoints 
checkpoint_files = sorted(
    weights_dir.glob("epoch*.pt"),
    key=lambda x: int(
        "".join(filter(str.isdigit, x.stem))
    )
)

print("\nCheckpoints found:")
for checkpoint in checkpoint_files:
    print(checkpoint.name)

### Evaluate on every checkpoint

In [ ]:
# import libs to calculate metrics
from modules.helper.calculate_metrics import calculate_metrics

In [ ]:
all_metrics = []

for index, checkpoint_path in enumerate(checkpoint_files):

    print(f"\nTesting: {checkpoint_path.name}")

    epoch_model = YOLO(str(checkpoint_path))

    y_true_train = []
    y_pred_train = []

    y_true_val = []
    y_pred_val = []

    # -----------------------
    # Evaluate Train Set
    # -----------------------
    for images, labels in train_loader:

        images_np = (
            images.permute(0, 2, 3, 1)
            .cpu()
            .numpy()
            * 255
        ).astype("uint8")

        predictions = epoch_model.predict(
            source=list(images_np),
            imgsz=IMAGE_SIZE,
            verbose=False
        )

        predicted_classes = [
            int(pred.probs.top1)
            for pred in predictions
        ]

        y_true_train.extend(labels.cpu().numpy())
        y_pred_train.extend(predicted_classes)


    # -----------------------
    # Evaluate Validation Set
    # -----------------------
    for images, labels in val_loader:

        images_np = (
            images.permute(0, 2, 3, 1)
            .cpu()
            .numpy()
            * 255
        ).astype("uint8")

        predictions = epoch_model.predict(
            source=list(images_np),
            imgsz=IMAGE_SIZE,
            verbose=False
        )

        predicted_classes = [
            int(pred.probs.top1)
            for pred in predictions
        ]

        y_true_val.extend(labels.cpu().numpy())
        y_pred_val.extend(predicted_classes)


    # Convert to tensors
    y_true_train = torch.tensor(y_true_train)
    y_pred_train = torch.tensor(y_pred_train)

    y_true_val = torch.tensor(y_true_val)
    y_pred_val = torch.tensor(y_pred_val)


    # Calculate metrics
    train_metrics = calculate_metrics(
        y_true_train,
        y_pred_train,
        prefix="train_"
    )

    val_metrics = calculate_metrics(
        y_true_val,
        y_pred_val,
        prefix="val_"
    )


    # Save epoch result
    all_metrics.append({
        "epoch": index + 1,
        "checkpoint": checkpoint_path.name,
        **train_metrics,
        **val_metrics
    })


# Convert to dataframe
metrics = pd.DataFrame(all_metrics)

metrics.head()

### Plot the line

In [ ]:
# import pre-developed module
from modules.helper.charts import gen_line_charts

In [ ]:
gen_line_charts(metrics, path, "training_metrics_graph.png", ["train_", "val_"])